<a href="https://colab.research.google.com/github/jaysulk/hartley-neural-operator/blob/main/HNO_FNO_run.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-

import os, gc, sys, json, pickle, time, argparse
import numpy as np
from datetime import datetime
import torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
from torch.utils.data import DataLoader, Dataset, Subset

try:
    from tqdm import tqdm
except Exception:
    def tqdm(x, **k): return x

QUICK = os.environ.get('QUICK', '0') == '1'

SAVE_DIR = '/content/drive/MyDrive/Allerton_experiments'
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception:
    SAVE_DIR = os.environ.get('SAVE_DIR', '/home/claude/exp_out')
os.makedirs(SAVE_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda':
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
print(f"Device: {device} | SAVE_DIR: {SAVE_DIR} | QUICK={QUICK}")

CONFIG = {
    'Nx': 64, 'Ny': 64, 'Nt': 51,
    'Nsamples': 200, 'Ntest': 40, 'batch_size': 4,
    'epochs': 150, 'eval_every': 25,
    'matern_nu': 2.5, 'matern_length_scale': 0.15, 'matern_sigma': 1.0,
    'seeds': [0],
    'unit_per_hour': 8.0,   # for ETA/budget reporting only
}
if QUICK:
    CONFIG.update(Nx=16, Ny=16, Nt=11, Nsamples=8, Ntest=4, batch_size=2,
                  epochs=2, eval_every=1, seeds=[0, 1])

IC_TYPES_ALL = ['eigenfunction', 'grf', 'gaussian_bump']
PDE_LIST_ALL = ['heat', 'wave', 'burgers', 'navier_stokes',
                'advection_diffusion', 'poisson', 'biharmonic']
BC_LIST_ALL = ['periodic', 'dirichlet']

PDE_PARAM_CASES = {
    'heat':                [{'nu_heat': 0.01,  'label': 'nu0.01'}],
    'wave':                [{'c_wave': 1.0,    'label': 'c1.0'}],
    'burgers':             [{'nu_burgers': 0.02, 'label': 'nu0.02'}],
    'navier_stokes':       [{'nu_ns': 0.005,  'label': 'nu0.005'}],
    'advection_diffusion': [{'c_adv': 1.5, 'nu_adv': 0.01, 'label': 'c1.5'}],
    'poisson':             [{'ls': 0.15, 'label': 'ls0.15'}],
    'biharmonic':          [{'ls': 0.15, 'label': 'ls0.15'}],
}

FIXED_HP = {
    'fno': {'modes': 10, 'width': 24, 'lr': 1e-3, 'weight_decay': 1e-4, 'grad_clip': 1.0, 'scheduler': 'step'},
    'hno': {'modes': 10, 'width': 24, 'lr': 1e-3, 'weight_decay': 1e-4, 'grad_clip': 1.0, 'scheduler': 'step'},
}
OP_MODES = FIXED_HP['fno']['modes']   # advection resolution guard uses this

RUN_PDES = PDE_LIST_ALL
RUN_BCS  = BC_LIST_ALL
RUN_ICS  = IC_TYPES_ALL

ELLIPTIC = {'poisson', 'biharmonic'}

def bc_window(Nx, Ny, device='cpu'):
    x = torch.linspace(0, 1, Nx, device=device); y = torch.linspace(0, 1, Ny, device=device)
    X, Y = torch.meshgrid(x, y, indexing='ij')
    return torch.sin(np.pi * X) * torch.sin(np.pi * Y)

def generate_eigenfunction_ic(Nx, Ny, n_modes=5, device='cpu'):
    x = torch.linspace(0, 2*np.pi, Nx, device=device); y = torch.linspace(0, 2*np.pi, Ny, device=device)
    X, Y = torch.meshgrid(x, y, indexing='ij'); ic = torch.zeros_like(X)
    for _ in range(n_modes):
        kx, ky = np.random.randint(1, 6), np.random.randint(1, 6)
        amp = np.random.randn() / (kx**2 + ky**2 + 1)
        ic += amp * torch.sin(kx*X + ky*Y + np.random.rand()*2*np.pi)
    return ic / (ic.abs().max() + 1e-8)

def generate_grf_ic(Nx, Ny, nu=2.5, length_scale=0.15, sigma=1.0, device='cpu'):
    kx = torch.fft.fftfreq(Nx, d=1.0/Nx, device=device); ky = torch.fft.fftfreq(Ny, d=1.0/Ny, device=device)
    KX, KY = torch.meshgrid(kx, ky, indexing='ij'); k_sq = KX**2 + KY**2
    tau = 2*nu / (length_scale**2)
    spectrum = (tau + 4*np.pi**2*k_sq)**(-(nu + 1)); spectrum[0, 0] = 0
    sqrt_spec = torch.sqrt(spectrum) * sigma * np.sqrt(Nx*Ny)
    noise = torch.complex(torch.randn(Nx, Ny, device=device), torch.randn(Nx, Ny, device=device))
    u = torch.fft.ifft2(sqrt_spec * noise).real
    return u / (u.abs().max() + 1e-8)

def generate_bump_ic(Nx, Ny, device='cpu'):
    x = torch.linspace(0, 1, Nx, device=device); y = torch.linspace(0, 1, Ny, device=device)
    X, Y = torch.meshgrid(x, y, indexing='ij'); u = torch.zeros(Nx, Ny, device=device)
    for _ in range(np.random.randint(2, 5)):
        cx, cy = np.random.uniform(0.2, 0.8), np.random.uniform(0.2, 0.8); sig = np.random.uniform(0.06, 0.15)
        u += np.random.uniform(0.5, 1.0) * torch.exp(-((X-cx)**2 + (Y-cy)**2)/(2*sig**2))
    return u / (u.abs().max() + 1e-8)

def get_ic_generator(ic_type, config, length_scale=None):
    Nx, Ny = config['Nx'], config['Ny']
    ls = length_scale if length_scale is not None else config['matern_length_scale']
    if ic_type == 'eigenfunction':
        return lambda: generate_eigenfunction_ic(Nx, Ny, device=device)
    if ic_type == 'grf':
        return lambda: generate_grf_ic(Nx, Ny, nu=config['matern_nu'], length_scale=ls,
                                       sigma=config['matern_sigma'], device=device)
    if ic_type == 'gaussian_bump':
        return lambda: generate_bump_ic(Nx, Ny, device=device)

def hartley3d(x):
    Xf = torch.fft.fftn(x, dim=[2,3,4]); return Xf.real - Xf.imag
def ihartley3d(H, shape):
    N = shape[0]*shape[1]*shape[2]; x_f = torch.fft.fftn(H, dim=[2,3,4]); return (x_f.real - x_f.imag)/N
def hartley2d(x):
    Xf = torch.fft.fft2(x, dim=[2,3]); return Xf.real - Xf.imag
def ihartley2d(H, shape):
    N = shape[0]*shape[1]; x_f = torch.fft.fft2(H, dim=[2,3]); return (x_f.real - x_f.imag)/N
def compl_mul3d(a, b): return torch.einsum("bixyz,ioxyz->boxyz", a, b)
def compl_mul2d(a, b): return torch.einsum("bixy,ioxy->boxy", a, b)
# (get_H_negative_* are unused by the clean real-diagonal HNO; kept for reference)
def get_H_negative_3d(H): return torch.roll(H.flip(dims=[2,3,4]), shifts=(1,1,1), dims=(2,3,4))
def get_H_negative_2d(H): return torch.roll(H.flip(dims=[2,3]), shifts=(1,1), dims=(2,3))

class SpectralConv2d_FNO(nn.Module):
    def __init__(self, in_ch, out_ch, modes):
        super().__init__(); self.in_ch, self.out_ch, self.modes = in_ch, out_ch, modes
        scale = 1.0/(in_ch+out_ch)
        self.w1 = nn.Parameter(scale*torch.rand(in_ch,out_ch,modes,modes,dtype=torch.cfloat))
        self.w2 = nn.Parameter(scale*torch.rand(in_ch,out_ch,modes,modes,dtype=torch.cfloat))
    def forward(self, x):
        B = x.shape[0]; x_ft = torch.fft.rfft2(x, dim=[2,3])
        out_ft = torch.zeros(B,self.out_ch,x.size(2),x.size(3)//2+1,device=x.device,dtype=torch.cfloat)
        m = self.modes
        out_ft[:,:,:m,:m]  = compl_mul2d(x_ft[:,:,:m,:m],  self.w1)
        out_ft[:,:,-m:,:m] = compl_mul2d(x_ft[:,:,-m:,:m], self.w2)
        return torch.fft.irfft2(out_ft, s=(x.size(2),x.size(3)), dim=[2,3])

class SpectralConv3d_FNO(nn.Module):
    def __init__(self, in_ch, out_ch, modes):
        super().__init__(); self.in_ch, self.out_ch, self.modes = in_ch, out_ch, modes
        scale = 1.0/(in_ch+out_ch)
        self.w1 = nn.Parameter(scale*torch.rand(in_ch,out_ch,modes,modes,modes,dtype=torch.cfloat))
        self.w2 = nn.Parameter(scale*torch.rand(in_ch,out_ch,modes,modes,modes,dtype=torch.cfloat))
        self.w3 = nn.Parameter(scale*torch.rand(in_ch,out_ch,modes,modes,modes,dtype=torch.cfloat))
        self.w4 = nn.Parameter(scale*torch.rand(in_ch,out_ch,modes,modes,modes,dtype=torch.cfloat))
    def forward(self, x):
        B = x.shape[0]; x_ft = torch.fft.rfftn(x, dim=[2,3,4]); m = self.modes
        out_ft = torch.zeros(B,self.out_ch,x.size(2),x.size(3),x.size(4)//2+1,device=x.device,dtype=torch.cfloat)
        out_ft[:,:,:m,:m,:m]   = compl_mul3d(x_ft[:,:,:m,:m,:m],   self.w1)
        out_ft[:,:,-m:,:m,:m]  = compl_mul3d(x_ft[:,:,-m:,:m,:m],  self.w2)
        out_ft[:,:,:m,-m:,:m]  = compl_mul3d(x_ft[:,:,:m,-m:,:m],  self.w3)
        out_ft[:,:,-m:,-m:,:m] = compl_mul3d(x_ft[:,:,-m:,-m:,:m], self.w4)
        return torch.fft.irfftn(out_ft, s=(x.size(2),x.size(3),x.size(4)), dim=[2,3,4])

class SpectralConv3d_HNO(nn.Module):
    def __init__(self, in_ch, out_ch, modes):
        super().__init__(); self.in_ch, self.out_ch, self.modes = in_ch, out_ch, modes
        s = 1.0/(in_ch+out_ch)
        self.w = nn.ParameterList([nn.Parameter(s*torch.rand(in_ch,out_ch,modes,modes,modes))
                                   for _ in range(8)])
    def forward(self, x):
        B,C,Nx,Ny,Nz = x.shape; m = self.modes
        H = hartley3d(x)
        out = torch.zeros(B,self.out_ch,Nx,Ny,Nz,device=x.device)
        sx=[slice(0,m),slice(-m,None)]; sy=[slice(0,m),slice(-m,None)]; sz=[slice(0,m),slice(-m,None)]
        i=0
        for a in sx:
            for b in sy:
                for c in sz:
                    sl=(slice(None),slice(None),a,b,c)
                    out[sl]=compl_mul3d(H[sl], self.w[i]); i+=1
        return ihartley3d(out, (Nx,Ny,Nz))

class SpectralConv2d_HNO(nn.Module):
    def __init__(self, in_ch, out_ch, modes):
        super().__init__(); self.in_ch, self.out_ch, self.modes = in_ch, out_ch, modes
        s = 1.0/(in_ch+out_ch)
        self.w_ll = nn.Parameter(s*torch.rand(in_ch,out_ch,modes,modes))
        self.w_lh = nn.Parameter(s*torch.rand(in_ch,out_ch,modes,modes))
        self.w_hl = nn.Parameter(s*torch.rand(in_ch,out_ch,modes,modes))
        self.w_hh = nn.Parameter(s*torch.rand(in_ch,out_ch,modes,modes))
    def forward(self, x):
        B,C,Nx,Ny = x.shape; m = self.modes
        H = hartley2d(x)
        out = torch.zeros(B,self.out_ch,Nx,Ny,device=x.device)
        out[:,:, :m, :m] = compl_mul2d(H[:,:, :m, :m], self.w_ll)
        out[:,:, :m,-m:] = compl_mul2d(H[:,:, :m,-m:], self.w_lh)
        out[:,:,-m:, :m] = compl_mul2d(H[:,:,-m:, :m], self.w_hl)
        out[:,:,-m:,-m:] = compl_mul2d(H[:,:,-m:,-m:], self.w_hh)
        return ihartley2d(out, (Nx,Ny))

class NeuralOperator3d(nn.Module):
    def __init__(self, modes, width=32, method='fno'):
        super().__init__(); self.width = width
        self.fc0 = nn.Linear(4, width)
        ConvClass = {'fno': SpectralConv3d_FNO, 'hno': SpectralConv3d_HNO}[method]
        self.conv0 = ConvClass(width, width, modes); self.conv1 = ConvClass(width, width, modes); self.conv2 = ConvClass(width, width, modes)
        self.w0 = nn.Conv1d(width, width, 1); self.w1 = nn.Conv1d(width, width, 1); self.w2 = nn.Conv1d(width, width, 1)
        self.fc1 = nn.Linear(width, 128); self.fc2 = nn.Linear(128, 1)
    def forward(self, x):
        B, nx, ny, nt = x.shape[:4]
        x = self.fc0(x).permute(0,4,1,2,3)
        x1=self.conv0(x); x2=self.w0(x.reshape(B,self.width,-1)).reshape(B,self.width,nx,ny,nt); x=F.gelu(x1+x2)
        x1=self.conv1(x); x2=self.w1(x.reshape(B,self.width,-1)).reshape(B,self.width,nx,ny,nt); x=F.gelu(x1+x2)
        x1=self.conv2(x); x2=self.w2(x.reshape(B,self.width,-1)).reshape(B,self.width,nx,ny,nt); x=x1+x2
        x = x.permute(0,2,3,4,1); return self.fc2(F.gelu(self.fc1(x)))

class NeuralOperator2d(nn.Module):
    def __init__(self, modes, width=32, method='fno'):
        super().__init__(); self.width = width
        self.fc0 = nn.Linear(3, width)
        ConvClass = {'fno': SpectralConv2d_FNO, 'hno': SpectralConv2d_HNO}[method]
        self.conv0=ConvClass(width,width,modes); self.conv1=ConvClass(width,width,modes); self.conv2=ConvClass(width,width,modes); self.conv3=ConvClass(width,width,modes)
        self.w0=nn.Conv1d(width,width,1); self.w1=nn.Conv1d(width,width,1); self.w2=nn.Conv1d(width,width,1); self.w3=nn.Conv1d(width,width,1)
        self.fc1=nn.Linear(width,128); self.fc2=nn.Linear(128,1)
    def forward(self, x):
        B, nx, ny, _ = x.shape
        x = self.fc0(x).permute(0,3,1,2)
        x1=self.conv0(x); x2=self.w0(x.reshape(B,self.width,-1)).reshape(B,self.width,nx,ny); x=F.gelu(x1+x2)
        x1=self.conv1(x); x2=self.w1(x.reshape(B,self.width,-1)).reshape(B,self.width,nx,ny); x=F.gelu(x1+x2)
        x1=self.conv2(x); x2=self.w2(x.reshape(B,self.width,-1)).reshape(B,self.width,nx,ny); x=F.gelu(x1+x2)
        x1=self.conv3(x); x2=self.w3(x.reshape(B,self.width,-1)).reshape(B,self.width,nx,ny); x=x1+x2
        x = x.permute(0,2,3,1); return self.fc2(F.gelu(self.fc1(x)))

class TimeDependentDataset(Dataset):
    def __init__(self, solutions):
        self.solutions = solutions; self.N, self.Nx, self.Ny, self.Nt = solutions.shape
        x=torch.linspace(0,1,self.Nx,device=solutions.device); y=torch.linspace(0,1,self.Ny,device=solutions.device); t=torch.linspace(0,1,self.Nt,device=solutions.device)
        self.X,self.Y,self.T = torch.meshgrid(x,y,t,indexing='ij')
    def __len__(self): return self.N
    def __getitem__(self, idx):
        u=self.solutions[idx]; u0=u[:,:,0:1].expand(-1,-1,self.Nt)
        return torch.stack([u0,self.X,self.Y,self.T],dim=-1), u.unsqueeze(0)

class EllipticDataset(Dataset):
    def __init__(self, sources, solutions):
        self.sources=sources; self.solutions=solutions; self.N,self.Nx,self.Ny = sources.shape
        x=torch.linspace(0,1,self.Nx,device=sources.device); y=torch.linspace(0,1,self.Ny,device=sources.device)
        self.X,self.Y = torch.meshgrid(x,y,indexing='ij')
    def __len__(self): return self.N
    def __getitem__(self, idx):
        return torch.stack([self.sources[idx],self.X,self.Y],dim=-1), self.solutions[idx].unsqueeze(0)

def _k(Nx, Ny, dev):
    kx=torch.fft.fftfreq(Nx,d=1.0/Nx,device=dev)*2*np.pi; ky=torch.fft.fftfreq(Ny,d=1.0/Ny,device=dev)*2*np.pi
    return torch.meshgrid(kx,ky,indexing='ij')

_ADV_RES_WARNED=set()
def _check_adv_resolution(ic, c, Nt, t_end=0.5):
    key=(round(float(c),3), int(Nt))
    if key in _ADV_RES_WARNED: return
    Nx,Ny=ic.shape; dt=t_end/(Nt-1); KX,KY=_k(Nx,Ny,ic.device)
    mag=torch.fft.fft2(ic).abs(); sig=mag>0.01*mag.max()
    if not sig.any(): return
    kmax=((KX+KY).abs()/(2*np.pi))[sig].max().item()        # ~ (kx+ky) of fastest populated mode
    step=(abs(c)*(KX+KY).abs())[sig].max().item()*dt
    msg=None
    if step>np.pi:
        msg=f"DATA-aliased (max temporal phase step {step:.2f} rad > pi); raise Nt or reduce c"
    elif OP_MODES is not None and 0.5*abs(c)*kmax > OP_MODES:
        cyc=0.5*abs(c)*kmax
        msg=f"exceeds OPERATOR temporal modes ({cyc:.1f} cyc > {OP_MODES}); reduce c or raise modes"
    if msg:
        _ADV_RES_WARNED.add(key)
        print(f"  [warn] advection under-resolved (c={c}, Nt={Nt}): {msg}.")

def solve_heat_periodic(ic, nu, Nt, t_end=0.5):
    KX,KY=_k(*ic.shape,ic.device); Ks=KX**2+KY**2; u0=torch.fft.fft2(ic)
    t=torch.linspace(0,t_end,Nt,device=ic.device)
    return torch.stack([torch.fft.ifft2(u0*torch.exp(-nu*Ks*tv)).real for tv in t], dim=-1)

def solve_wave_periodic(ic, c, Nt, t_end=0.5):
    KX,KY=_k(*ic.shape,ic.device); Km=torch.sqrt(KX**2+KY**2); u0=torch.fft.fft2(ic)
    t=torch.linspace(0,t_end,Nt,device=ic.device)
    return torch.stack([torch.fft.ifft2(u0*torch.cos(c*Km*tv)).real for tv in t], dim=-1)

def solve_advdiff_periodic(ic, c, nu, Nt, t_end=0.5):
    _check_adv_resolution(ic, c, Nt, t_end)
    KX,KY=_k(*ic.shape,ic.device); decay=nu*(KX**2+KY**2); phase=c*(KX+KY); u0=torch.fft.fft2(ic)
    t=torch.linspace(0,t_end,Nt,device=ic.device)
    return torch.stack([torch.fft.ifft2(u0*torch.exp(-decay*tv)*torch.exp(-1j*phase*tv)).real for tv in t], dim=-1)

def solve_burgers_periodic(ic, nu, Nt, t_end=0.5):
    Nx,Ny=ic.shape; dx=1.0/(Nx-1); dt_out=t_end/(Nt-1); cur=ic.clone(); U=[ic.clone()]
    for _ in range(Nt-1):
        tr=dt_out; s=0
        while tr>1e-14 and s<10000:
            um=cur.abs().max().item()+1e-8; dt=min(0.3*dx/um,0.2*dx**2/(nu+1e-8),tr)
            uxb=(cur-torch.roll(cur,1,0))/dx; uxf=(torch.roll(cur,-1,0)-cur)/dx
            uyb=(cur-torch.roll(cur,1,1))/dx; uyf=(torch.roll(cur,-1,1)-cur)/dx
            ux=torch.where(cur>=0,uxb,uxf); uy=torch.where(cur>=0,uyb,uyf)
            lap=(torch.roll(cur,-1,0)+torch.roll(cur,1,0)+torch.roll(cur,-1,1)+torch.roll(cur,1,1)-4*cur)/dx**2
            cur=(cur+dt*(-cur*ux-cur*uy+nu*lap)).clamp(-5,5); tr-=dt; s+=1
        U.append(cur.clone())
    return torch.stack(U[:Nt],dim=-1)

def solve_ns_periodic(ic, nu, Nt, t_end=0.5):
    Nx,Ny=ic.shape; dx=1.0/(Nx-1); KX,KY=_k(Nx,Ny,ic.device); K2=KX**2+KY**2; K2[0,0]=1
    dt_out=t_end/(Nt-1); cur=ic.clone(); U=[ic.clone()]
    for _ in range(Nt-1):
        tr=dt_out
        while tr>1e-14:
            wh=torch.fft.fft2(cur); ph=-wh/K2; ph[0,0]=0; psi=torch.fft.ifft2(ph).real
            uv=(torch.roll(psi,-1,1)-torch.roll(psi,1,1))/(2*dx); vv=-(torch.roll(psi,-1,0)-torch.roll(psi,1,0))/(2*dx)
            um=max(uv.abs().max().item(),vv.abs().max().item())+1e-8; dt=min(0.3*dx/um,0.2*dx**2/(nu+1e-8),tr)
            wxb=(cur-torch.roll(cur,1,0))/dx; wxf=(torch.roll(cur,-1,0)-cur)/dx
            wyb=(cur-torch.roll(cur,1,1))/dx; wyf=(torch.roll(cur,-1,1)-cur)/dx
            wx=torch.where(uv>=0,wxb,wxf); wy=torch.where(vv>=0,wyb,wyf)
            lap=(torch.roll(cur,-1,0)+torch.roll(cur,1,0)+torch.roll(cur,-1,1)+torch.roll(cur,1,1)-4*cur)/dx**2
            cur=cur+dt*(-uv*wx-vv*wy+nu*lap); tr-=dt
        U.append(cur.clone())
    return torch.stack(U[:Nt],dim=-1)

def solve_poisson_periodic(source):
    KX,KY=_k(*source.shape,source.device); Ks=KX**2+KY**2; Ks[0,0]=1
    uh=torch.fft.fft2(source)/Ks; uh[0,0]=0; return torch.fft.ifft2(uh).real

def solve_biharmonic_periodic(source):
    KX,KY=_k(*source.shape,source.device); K4=(KX**2+KY**2)**2; K4[0,0]=1
    uh=torch.fft.fft2(source)/K4; uh[0,0]=0; return torch.fft.ifft2(uh).real

def _lap_dir(u, dx):
    return (u[2:,1:-1]+u[:-2,1:-1]+u[1:-1,2:]+u[1:-1,:-2]-4*u[1:-1,1:-1])/dx**2
def _zero_bc(u): u[0,:]=0; u[-1,:]=0; u[:,0]=0; u[:,-1]=0; return u

def solve_heat_dirichlet(ic, nu, Nt, t_end=0.5):
    Nx,Ny=ic.shape; dx=1.0/(Nx-1); dt_out=t_end/(Nt-1); u=_zero_bc(ic.clone()); U=[u.clone()]
    for _ in range(Nt-1):
        tr=dt_out
        while tr>1e-14:
            dt=min(0.2*dx**2/(nu+1e-8),tr); un=u.clone()
            un[1:-1,1:-1]=u[1:-1,1:-1]+dt*nu*_lap_dir(u,dx); u=_zero_bc(un); tr-=dt
        U.append(u.clone())
    return torch.stack(U,dim=-1)

def solve_wave_dirichlet(ic, c, Nt, t_end=0.5):
    Nx,Ny=ic.shape; dx=1.0/(Nx-1); dt_out=t_end/(Nt-1); u0=_zero_bc(ic.clone())
    dt_cfl=0.5*dx/(c*np.sqrt(2)+1e-8)
    u_prev=u0.clone(); u_curr=u0.clone()
    u_curr[1:-1,1:-1]=u0[1:-1,1:-1]+0.5*(c*dt_cfl)**2*_lap_dir(u0,dx); u_curr=_zero_bc(u_curr)
    saved=[u0.clone()]; t_acc=0.0; t_targets=[i*dt_out for i in range(1,Nt)]; ti=0
    while ti < len(t_targets):
        un=u_curr.clone()
        un[1:-1,1:-1]=2*u_curr[1:-1,1:-1]-u_prev[1:-1,1:-1]+(c*dt_cfl)**2*_lap_dir(u_curr,dx)
        un=_zero_bc(un); u_prev=u_curr; u_curr=un; t_acc+=dt_cfl
        while ti<len(t_targets) and t_acc>=t_targets[ti]-1e-12:
            saved.append(u_curr.clone()); ti+=1
    return torch.stack(saved[:Nt],dim=-1)

def solve_advdiff_dirichlet(ic, c, nu, Nt, t_end=0.5):
    _check_adv_resolution(ic, c, Nt, t_end)
    Nx,Ny=ic.shape; dx=1.0/(Nx-1); dt_out=t_end/(Nt-1); u=_zero_bc(ic.clone()); U=[u.clone()]
    for _ in range(Nt-1):
        tr=dt_out
        while tr>1e-14:
            dt=min(0.4*dx/(abs(c)*np.sqrt(2)+1e-8), 0.2*dx**2/(nu+1e-8), tr)
            ui=u[1:-1,1:-1]
            uxb=(ui-u[:-2,1:-1])/dx; uyb=(ui-u[1:-1,:-2])/dx
            lap=_lap_dir(u,dx); un=u.clone()
            un[1:-1,1:-1]=ui+dt*(-c*uxb - c*uyb + nu*lap); u=_zero_bc(un); tr-=dt
        U.append(u.clone())
    return torch.stack(U,dim=-1)

def solve_burgers_dirichlet(ic, nu, Nt, t_end=0.5):
    Nx,Ny=ic.shape; dx=1.0/(Nx-1); dt_out=t_end/(Nt-1); u=_zero_bc(ic.clone()); U=[u.clone()]
    for _ in range(Nt-1):
        tr=dt_out; s=0
        while tr>1e-14 and s<10000:
            um=u.abs().max().item()+1e-8; dt=min(0.3*dx/um,0.2*dx**2/(nu+1e-8),tr); ui=u[1:-1,1:-1]
            uxb=(ui-u[:-2,1:-1])/dx; uxf=(u[2:,1:-1]-ui)/dx
            uyb=(ui-u[1:-1,:-2])/dx; uyf=(u[1:-1,2:]-ui)/dx
            ux=torch.where(ui>=0,uxb,uxf); uy=torch.where(ui>=0,uyb,uyf); lap=_lap_dir(u,dx)
            un=u.clone(); un[1:-1,1:-1]=(ui+dt*(-ui*ux-ui*uy+nu*lap)).clamp(-5,5); u=_zero_bc(un); tr-=dt; s+=1
        U.append(u.clone())
    return torch.stack(U[:Nt],dim=-1)

def _dst2_solve(rhs, eig):
    Nx,Ny=rhs.shape; dev=rhs.device
    idx=torch.arange(1,Nx-1,device=dev).float()
    S=torch.sin(np.pi*torch.outer(idx,idx)/(Nx-1))*np.sqrt(2.0/(Nx-1))
    b=rhs[1:-1,1:-1]; bhat=S@b@S.T; uhat=bhat/eig; uint=S@uhat@S.T
    u=torch.zeros(Nx,Ny,device=dev); u[1:-1,1:-1]=uint; return u

def _dir_eig(Nx, dx):
    idx=torch.arange(1,Nx-1).float(); return (2-2*torch.cos(np.pi*idx/(Nx-1)))/dx**2

def solve_poisson_dirichlet(source):
    Nx,Ny=source.shape; dx=1.0/(Nx-1)
    lam=_dir_eig(Nx,dx).to(source.device); eig=lam[:,None]+lam[None,:]
    return _dst2_solve(source, eig)

def solve_biharmonic_dirichlet(source):
    Nx,Ny=source.shape; dx=1.0/(Nx-1)
    lam=_dir_eig(Nx,dx).to(source.device); eig=(lam[:,None]+lam[None,:])**2
    return _dst2_solve(source, eig)

def solve_ns_dirichlet(ic, nu, Nt, t_end=0.5):
    Nx,Ny=ic.shape; dx=1.0/(Nx-1); dt_out=t_end/(Nt-1)
    lam=_dir_eig(Nx,dx).to(ic.device); eig=lam[:,None]+lam[None,:]
    cur=_zero_bc(ic.clone()); U=[cur.clone()]
    for _ in range(Nt-1):
        tr=dt_out
        while tr>1e-14:
            psi=_dst2_solve(cur, eig)
            uv=torch.zeros_like(psi); vv=torch.zeros_like(psi)
            uv[1:-1,1:-1]=(psi[1:-1,2:]-psi[1:-1,:-2])/(2*dx)
            vv[1:-1,1:-1]=-(psi[2:,1:-1]-psi[:-2,1:-1])/(2*dx)
            um=max(uv.abs().max().item(),vv.abs().max().item())+1e-8; dt=min(0.3*dx/um,0.2*dx**2/(nu+1e-8),tr)
            ci=cur[1:-1,1:-1]
            wxb=(ci-cur[:-2,1:-1])/dx; wxf=(cur[2:,1:-1]-ci)/dx
            wyb=(ci-cur[1:-1,:-2])/dx; wyf=(cur[1:-1,2:]-ci)/dx
            wx=torch.where(uv[1:-1,1:-1]>=0,wxb,wxf); wy=torch.where(vv[1:-1,1:-1]>=0,wyb,wyf)
            lap=_lap_dir(cur,dx); un=cur.clone()
            un[1:-1,1:-1]=ci+dt*(-uv[1:-1,1:-1]*wx-vv[1:-1,1:-1]*wy+nu*lap); cur=_zero_bc(un); tr-=dt
        U.append(cur.clone())
    return torch.stack(U[:Nt],dim=-1)

def time_solver(pde, bc):
    table = {
        ('heat','periodic'):        lambda ic,p,Nt: solve_heat_periodic(ic,p['nu_heat'],Nt),
        ('heat','dirichlet'):       lambda ic,p,Nt: solve_heat_dirichlet(ic,p['nu_heat'],Nt),
        ('wave','periodic'):        lambda ic,p,Nt: solve_wave_periodic(ic,p['c_wave'],Nt),
        ('wave','dirichlet'):       lambda ic,p,Nt: solve_wave_dirichlet(ic,p['c_wave'],Nt),
        ('burgers','periodic'):     lambda ic,p,Nt: solve_burgers_periodic(ic,p['nu_burgers'],Nt),
        ('burgers','dirichlet'):    lambda ic,p,Nt: solve_burgers_dirichlet(ic,p['nu_burgers'],Nt),
        ('navier_stokes','periodic'):  lambda ic,p,Nt: solve_ns_periodic(ic,p['nu_ns'],Nt),
        ('navier_stokes','dirichlet'): lambda ic,p,Nt: solve_ns_dirichlet(ic,p['nu_ns'],Nt),
        ('advection_diffusion','periodic'):  lambda ic,p,Nt: solve_advdiff_periodic(ic,p['c_adv'],p['nu_adv'],Nt),
        ('advection_diffusion','dirichlet'): lambda ic,p,Nt: solve_advdiff_dirichlet(ic,p['c_adv'],p['nu_adv'],Nt),
    }
    return table[(pde,bc)]

def elliptic_solver(pde, bc):
    return {('poisson','periodic'): solve_poisson_periodic, ('poisson','dirichlet'): solve_poisson_dirichlet,
            ('biharmonic','periodic'): solve_biharmonic_periodic, ('biharmonic','dirichlet'): solve_biharmonic_dirichlet}[(pde,bc)]

def gen_data(pde, bc, params, ic_type, config, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    Nx,Ny,Nt = config['Nx'],config['Ny'],config['Nt']
    win = bc_window(Nx,Ny,device) if bc=='dirichlet' else None
    gen_ic = get_ic_generator(ic_type, config, length_scale=params.get('ls', None))
    if pde in ELLIPTIC:
        solver = elliptic_solver(pde, bc); src,sol=[],[]
        for _ in tqdm(range(config['Nsamples']), desc=f"  {pde}/{bc}/{ic_type}"):
            f=gen_ic()
            if win is not None: f=f*win
            src.append(f); sol.append(solver(f))
        S=torch.stack(src); So=torch.stack(sol)
        return ('elliptic', S/(S.abs().max()+1e-8), So/(So.abs().max()+1e-8))
    else:
        solver = time_solver(pde, bc); sols=[]
        for _ in tqdm(range(config['Nsamples']), desc=f"  {pde}/{bc}/{ic_type}"):
            ic=gen_ic(); ic=ic/(ic.abs().max()+1e-8)
            if win is not None: ic=ic*win
            u=solver(ic, params, Nt)
            if torch.isnan(u).any() or torch.isinf(u).any():
                u=solver(ic*0.5, params, Nt)
                if torch.isnan(u).any(): u=torch.zeros(Nx,Ny,Nt,device=device)
            sols.append(u)
        sols=torch.stack(sols); return ('time', sols/(sols.abs().max()+1e-8))

def build_loaders(packed, config):
    ds = EllipticDataset(packed[1], packed[2]) if packed[0]=='elliptic' else TimeDependentDataset(packed[1])
    ntr=config['Nsamples']-config['Ntest']
    tr=DataLoader(Subset(ds,range(ntr)), batch_size=config['batch_size'], shuffle=True)
    te=DataLoader(Subset(ds,range(ntr,config['Nsamples'])), batch_size=config['batch_size'])
    return tr,te

def real_param_count(model):
    return sum(p.numel()*(2 if torch.is_complex(p) else 1) for p in model.parameters())

def train_model(model, train_loader, test_loader, hp, epochs, is_3d, eval_every=25, verbose=True):
    lr=hp.get('lr',1e-3); wd=hp.get('weight_decay',1e-4); sched_type=hp.get('scheduler','step'); clip=hp.get('grad_clip',1.0)
    opt=optim.Adam(model.parameters(),lr=lr,weight_decay=wd)
    sch=optim.lr_scheduler.StepLR(opt,step_size=max(1,epochs//4),gamma=0.5) if sched_type=='step' else optim.lr_scheduler.CosineAnnealingLR(opt,T_max=epochs)
    perm=(lambda p: p.permute(0,4,1,2,3) if is_3d else p.permute(0,3,1,2))
    train_losses,test_curve=[],[]; t0=time.time()
    def evaluate():
        model.eval(); errs=[]
        with torch.no_grad():
            for inp,tgt in test_loader:
                inp,tgt=inp.to(device),tgt.to(device); pred=perm(model(inp))
                for i in range(pred.shape[0]): errs.append((torch.norm(pred[i]-tgt[i])/torch.norm(tgt[i])).item())
        return errs
    for ep in range(1,epochs+1):
        model.train(); el,nb=0.0,0
        for inp,tgt in train_loader:
            inp,tgt=inp.to(device),tgt.to(device); opt.zero_grad()
            pred=perm(model(inp))
            loss=((pred-tgt).flatten(1).norm(dim=1)/(tgt.flatten(1).norm(dim=1)+1e-8)).mean(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),clip); opt.step(); el+=loss.item(); nb+=1
        sch.step(); train_losses.append(el/nb)
        if ep%eval_every==0:
            test_curve.append((ep,float(np.mean(evaluate()))))
            if verbose: print(f"        Ep{ep}: train={train_losses[-1]:.5f} L2={test_curve[-1][1]:.4f}")
    model.eval(); ps2, ps1 = [], []
    with torch.no_grad():
        for inp,tgt in test_loader:
            inp,tgt=inp.to(device),tgt.to(device); pred=perm(model(inp))
            for i in range(pred.shape[0]):
                e=pred[i]-tgt[i]
                ps2.append((torch.norm(e)/(torch.norm(tgt[i])+1e-12)).item())
                ps1.append((e.abs().sum()/(tgt[i].abs().sum()+1e-12)).item())
    return {'l2_mean':float(np.mean(ps2)),'l2_std':float(np.std(ps2)),
            'l1_mean':float(np.mean(ps1)),'l1_std':float(np.std(ps1)),
            'per_sample_l2':ps2,'train_losses':train_losses,'test_l2_curve':test_curve,
            'train_time_sec':time.time()-t0}


def _field_2d(t, is_3d):
    """Squeeze [B,1,Nx,Ny(,Nt)] -> [Nx,Ny] numpy; final-time slice for 3d."""
    t = t[0, 0]
    if is_3d: t = t[..., -1]
    return t.detach().cpu().numpy()

def run_cell(pde, bc, ic_type, params, seed, hp_fno, hp_hno, config):
    is_3d = pde not in ELLIPTIC
    packed = gen_data(pde, bc, params, ic_type, config, seed)
    tr,te = build_loaders(packed, config); MC = NeuralOperator3d if is_3d else NeuralOperator2d
    perm = (lambda p: p.permute(0,4,1,2,3) if is_3d else p.permute(0,3,1,2))
    inp0, tgt0 = te.dataset[0]
    inp0 = inp0[None].to(device); tgt0 = tgt0[None].to(device)
    out={}; fields={'target': _field_2d(tgt0, is_3d)}
    for method,hp in (('fno',hp_fno),('hno',hp_hno)):
        torch.manual_seed(seed); np.random.seed(seed)
        m=MC(hp['modes'],hp['width'],method).to(device); n=real_param_count(m)
        r=train_model(m,tr,te,hp,config['epochs'],is_3d,eval_every=config['eval_every'],verbose=not QUICK)
        r['n_params']=n; out[method]=r
        m.eval()
        with torch.no_grad(): fields[method] = _field_2d(perm(m(inp0)), is_3d)
        del m; (torch.cuda.empty_cache() if device.type=='cuda' else None); gc.collect()
    out['fields']=fields
    del packed,tr,te; gc.collect()
    return out

def load_done(path):
    done=set()
    if os.path.exists(path):
        for line in open(path):
            try:
                r=json.loads(line); done.add((r['pde'],r['bc'],r['ic'],r['label'],r['seed']))
            except Exception: pass
    return done

def estimate_cost(done):
    cells = sum(len(PDE_PARAM_CASES[p]) for p in RUN_PDES) * len(RUN_BCS) * len(RUN_ICS) * len(CONFIG['seeds'])
    remaining = cells - len(done)
    eta_h = remaining * 400 / 3600.0
    units = eta_h * CONFIG['unit_per_hour']
    print(f"[PLAN] cells total={cells}  done={len(done)}  remaining={remaining}  "
          f"~ETA {eta_h:.1f}h  ~{units:.0f} units (@{CONFIG['unit_per_hour']}/h, 400s/cell est)")
    print(f"       grid: PDES={RUN_PDES} BCS={RUN_BCS} ICS={RUN_ICS} seeds={CONFIG['seeds']} epochs={CONFIG['epochs']}")

def main():
    jsonl=os.path.join(SAVE_DIR,'results_fig.jsonl')
    fdir=os.path.join(SAVE_DIR,'fields'); os.makedirs(fdir, exist_ok=True)
    done=load_done(jsonl); estimate_cost(done)
    hp_fno, hp_hno = FIXED_HP['fno'], FIXED_HP['hno']
    print(f"FNO HP: {hp_fno}\nHNO HP: {hp_hno}")
    f=open(jsonl,'a')
    t_run=time.time(); n_run=0
    for pde in RUN_PDES:
        for bc in RUN_BCS:
            #if (pde, bc) in SKIP_COMBOS:
                #print(f"  skip-combo {pde}/{bc} (degenerate)"); continue
            for ic in RUN_ICS:
                for params in PDE_PARAM_CASES[pde]:
                    for seed in CONFIG['seeds']:
                        key=(pde,bc,ic,params['label'],seed)
                        field_path=os.path.join(fdir, f"{pde}__{bc}__{ic}__s{seed}.npz")
                        if key in done and os.path.exists(field_path):
                            print(f"  skip {'/'.join(map(str,key))}"); continue
                        t0=time.time()
                        cell=run_cell(pde,bc,ic,params,seed,hp_fno,hp_hno,CONFIG)
                        fl,hl=cell['fno']['l2_mean'],cell['hno']['l2_mean']
                        f1,h1=cell['fno']['l1_mean'],cell['hno']['l1_mean']
                        fld=cell['fields']
                        np.savez_compressed(field_path,
                                            target=fld['target'], fno=fld['fno'], hno=fld['hno'],
                                            fno_l2=fl, hno_l2=hl, fno_l1=f1, hno_l1=h1)
                        row={'pde':pde,'bc':bc,'ic':ic,'label':params['label'],'seed':seed,
                             'fno':fl,'hno':hl,'fno_l1':f1,'hno_l1':h1,
                             'fno_std':cell['fno']['l2_std'],'hno_std':cell['hno']['l2_std'],
                             'n_fno':cell['fno']['n_params'],'n_hno':cell['hno']['n_params'],
                             'sec':round(time.time()-t0,1),'ts':datetime.now().isoformat()}
                        f.write(json.dumps(row)+'\n'); f.flush(); done.add(key); n_run+=1
                        units=CONFIG['unit_per_hour']*(time.time()-t_run)/3600
                        print(f"  {pde}/{bc}/{ic}/s{seed}: FNO={fl:.4f} HNO={hl:.4f} "
                              f"ratio={hl/(fl+1e-8):.2f}x ({row['sec']:.0f}s) [run={n_run} ~{units:.1f}u]")
    f.close()

In [ ]:
# ============================== CONTROL PANEL ==============================
RUN_PDES = ['advection_diffusion']   # all 7
RUN_BCS  = ['dirichlet']                          # both
RUN_ICS  = ['eigenfunction', 'grf', 'gaussian_bump']          # all 3

CONFIG['seeds']  = [0,1,2]          # ONE seed/cell
CONFIG['epochs'] = 150
SAVE_DIR = '/content/drive/MyDrive/TLMR_HNO_experiments'    # uncomment to change folder
# ==========================================================================
print(f"grid -> PDES={RUN_PDES} BCS={RUN_BCS} ICS={RUN_ICS} seeds={CONFIG['seeds']} epochs={CONFIG['epochs']}")
main()
